# 05 - Level-only baseline

Clusters the days with only two numbers per day: the mean and the standard deviation of that day's price (in logs for `real_price_log`). No scalograms, no CWT and no VAE. It is the same clustering input as notebooks 02 and 03 without the latent vector.

It answers the main question of the thesis: does the intraday shape of the price, which the CWT and the VAE capture, add information about the market regime beyond how expensive and how volatile the day was? A model only adds value if it beats this baseline in persistence (fewer switches per year) and in how well its most expensive regime captures the El Nino days of 2015-2016 and 2023-2024.

Pipeline: daily price level (mean, std) -> k-means or GMM -> regime

There are two formal baselines. With `level_window = 1` each day uses its own mean and standard deviation. With `level_window = 7` both are replaced by their backward 7-day mean (the day and the 6 days before it, so only past days are used), which removes most of the day-to-day noise and sets the persistence a model with time context should reach.

Runs are saved like any other run, in `experiments/level_only/<clustering>/baseline/` (the 7-day runs with `tag = "7d"`), with `study = "baseline"`, so notebook 04 can compare them with any model. Nothing is trained, so a full run takes about a minute (most of it is the t-SNE).

<d.benavidess@uniandes.edu.co>

In [1]:
import sys
sys.path.append("..")

from pathlib import Path

import numpy as np
import pandas as pd
import altair as alt
from sklearn.manifold import TSNE

alt.data_transformers.enable('vegafusion')

from src.data import load_context, chronological_split
from src.clustering import with_level, level_columns, select_k
from src.plots import selection_chart
from src.runs import prepare_runs, save_all, cluster_runs, k_sweep_table, tsne_grid, regime_strips
from src import experiments as exp

In [2]:
import warnings
warnings.filterwarnings("ignore")

## 0. Configuration

`scalogram_set` only decides which daily level is used: `"real_price_log"` (log_price_mean, log_price_std) or `"real_price"` (price_mean, price_std in COP/kWh). The scalogram images are not read.

`level_window` is the number of days in the backward mean of the daily mean and standard deviation: 1 for the daily baseline, 7 for the 7-day baseline (with `tag = "7d"`). The first days of the series average the days available.

In [3]:
scalogram_set = "real_price_log"
clustering = "kmeans"
k_values = [2, 3, 4, 5, 6]
study = "baseline"
level_window = 7
tag = "7d"

config = {
    "model": "level_only",
    "study": study,
    "tag": tag,
    "scalogram_set": scalogram_set,
    "clustering": clustering,
    "k_values": k_values,
    "k_range": [2, 3, 4, 5, 6, 7, 8],
    "features": "daily level (mean, std)",
    "level_window": level_window,
    "train_ratio": 0.6,
    "val_ratio": 0.2,
    "seed": 42,
}

## 1. Run folders

In [4]:
config, run_dirs, run_configs, _ = prepare_runs(config)
for k, run_dir in run_dirs.items():
    print(f"k={k}: {exp.run_id(run_dir)}")

k=2: level_only/kmeans/baseline/006_real_price_log_kmeans_k2_7d
k=3: level_only/kmeans/baseline/007_real_price_log_kmeans_k3_7d
k=4: level_only/kmeans/baseline/008_real_price_log_kmeans_k4_7d
k=5: level_only/kmeans/baseline/009_real_price_log_kmeans_k5_7d
k=6: level_only/kmeans/baseline/010_real_price_log_kmeans_k6_7d


## 2. Data

In [5]:
scalogram_dir = Path("../data/processed/scalograms") / config["scalogram_set"]
data_dir = Path("../data/processed")

metadata = pd.read_csv(scalogram_dir / "scalogram_metadata.csv", parse_dates=["start_datetime", "end_datetime"])
context = load_context(data_dir, metadata)
train_idx, val_idx, test_idx = chronological_split(len(metadata), config["train_ratio"], config["val_ratio"])
split_dates = [context["date"].iloc[val_idx[0]], context["date"].iloc[test_idx[0]]]

splits = exp.split_table(context, {"train": train_idx, "val": val_idx, "test": test_idx})
print(splits)

level_cols = list(level_columns(metadata))
level = metadata.copy()
level[level_cols] = metadata[level_cols].rolling(config["level_window"], min_periods=1).mean()
features = with_level(np.zeros((len(metadata), 0)), level, train_idx)
print(f"Level columns: {level_cols}, backward mean over {config['level_window']} day(s), z-scored on training days")
print(f"Features: {features.shape}, max z-score {features.max():.1f}")

+-------+------+------------+------------+
| Split | Days |    From    |     To     |
+-------+------+------------+------------+
| train | 5825 | 2000-01-01 | 2015-12-12 |
|  val  | 1942 | 2015-12-13 | 2021-04-06 |
|  test | 1942 | 2021-04-07 | 2026-07-31 |
+-------+------+------------+------------+
Level columns: ['log_price_mean', 'log_price_std'], backward mean over 7 day(s), z-scored on training days
Features: (9709, 2), max z-score 6.3


## 3. Choice of k

Same table as notebooks 02 and 03, on the two level columns only.

In [6]:
selection = select_k(features, train_idx, config["clustering"], config["k_range"], config["seed"])
selection_table = exp.selection_table(selection)
print(selection_table)
save_all(run_dirs, exp.save_table, "k_selection", selection_table, show=False)
selection_chart(selection, config["k_values"][0])

+---+---------+------------+----------------+-------------------+
| k | Inertia | Silhouette | Davies-Bouldin | Calinski-Harabasz |
+---+---------+------------+----------------+-------------------+
| 2 |  6,814  |   0.417    |     0.940      |       4,133       |
| 3 |  4,423  |   0.400    |     0.860      |       4,756       |
| 4 |  3,543  |   0.390    |     0.851      |       4,440       |
| 5 |  2,747  |   0.378    |     0.821      |       4,715       |
| 6 |  2,282  |   0.386    |     0.793      |       4,778       |
| 7 |  1,939  |   0.374    |     0.828      |       4,855       |
| 8 |  1,692  |   0.380    |     0.806      |       4,892       |
+---+---------+------------+----------------+-------------------+


alt.HConcatChart(...)

## 4. One run per k

The t-SNE map is made on the same two level columns so the figure is comparable with the model runs. The feature checks add the 7-day mean of the level, to show how much of the day-to-day switching smoothing removes.

In [7]:
rng = np.random.default_rng(config["seed"])
plot_idx = np.sort(rng.choice(len(features), size=min(5000, len(features)), replace=False))
coords = TSNE(n_components=2, perplexity=30, random_state=config["seed"]).fit_transform(features[plot_idx])

metadata_7d = metadata.copy()
metadata_7d[level_cols] = metadata[level_cols].rolling(7, min_periods=1).mean()
feature_sets = {
    "level only": with_level(np.zeros((len(metadata), 0)), metadata, train_idx),
    "level 7d": with_level(np.zeros((len(metadata), 0)), metadata_7d, train_idx),
}
results = cluster_runs(run_dirs, run_configs, features, context, train_idx, split_dates, selection,
                       coords, plot_idx, feature_sets)

k=2: level_only/kmeans/baseline/006_real_price_log_kmeans_k2_7d  silhouette=0.347  mean run=26.7 d  switches/yr=13.6
k=3: level_only/kmeans/baseline/007_real_price_log_kmeans_k3_7d  silhouette=0.384  mean run=23.7 d  switches/yr=15.3
k=4: level_only/kmeans/baseline/008_real_price_log_kmeans_k4_7d  silhouette=0.370  mean run=19.7 d  switches/yr=18.5
k=5: level_only/kmeans/baseline/009_real_price_log_kmeans_k5_7d  silhouette=0.344  mean run=14.2 d  switches/yr=25.6
k=6: level_only/kmeans/baseline/010_real_price_log_kmeans_k6_7d  silhouette=0.351  mean run=11.4 d  switches/yr=31.9


In [8]:
tsne_grid(coords, plot_idx, results)

alt.VConcatChart(...)

## 5. Comparison across k

Same columns as the k sweep of notebooks 02 and 03. Read the El Nino columns against `top_share_pct`.

In [9]:
k_sweep = k_sweep_table(results)
save_all(run_dirs, exp.save_table, "k_sweep", k_sweep, show=False)
k_sweep

,silhouette,davies_bouldin,calinski_harabasz,mean_run_days,switches_per_year,top_share_pct,top_price,top_reservoir,nino_2015_16_in_top_pct,nino_2023_24_in_top_pct
k,,,,,,,,,,
2,0.347,1.093,4767.432,26.747,13.618,67.916,369.890,0.653,97.872,88.955
3,0.384,0.841,7379.300,23.738,15.349,34.123,544.178,0.616,91.103,90.448
4,0.370,0.856,7612.226,19.654,18.547,9.702,953.139,0.590,41.393,74.627
5,0.344,0.886,7367.314,14.236,25.619,8.446,1007.136,0.590,41.006,67.761
6,0.351,0.888,7174.078,11.422,31.939,7.282,1065.042,0.588,39.845,57.910


In [10]:
for k, r in results.items():
    print(f"k={k}  {exp.run_id(run_dirs[k])}")
    print(exp.regime_profile_table(r["profile"]))
    print("Share of each weekday in each regime (%)")
    print(r["weekday"].to_string(), end="\n\n")

k=2  level_only/kmeans/baseline/006_real_price_log_kmeans_k2_7d
+--------+------+---------+------------+-----------+----------+-----------+
| Regime | Days | Share % | Real price | Reservoir | Mean ONI | El Nino % |
+--------+------+---------+------------+-----------+----------+-----------+
|   0    | 3115 |   32.1  |   202.1    |   0.733   |  -0.27   |    9.2    |
|   1    | 6594 |   67.9  |   369.9    |   0.653   |   0.13   |    32.1   |
+--------+------+---------+------------+-----------+----------+-----------+
Share of each weekday in each regime (%)
         Mon   Tue   Wed   Thu   Fri   Sat   Sun
regime                                          
0       32.0  31.0  32.0  33.0  32.0  32.0  32.0
1       68.0  69.0  68.0  67.0  68.0  68.0  68.0

k=3  level_only/kmeans/baseline/007_real_price_log_kmeans_k3_7d
+--------+------+---------+------------+-----------+----------+-----------+
| Regime | Days | Share % | Real price | Reservoir | Mean ONI | El Nino % |
+--------+------+---------

Regime strip for every k, one row per k.

In [11]:
regime_strips(results)

alt.VConcatChart(...)